In [1]:
import time

from omnes_pro_uno.fphse import Fphse

OP_ADD = b"\x01"

In [2]:
data_owner_num = 500
val_len = 47

In [3]:
fphse = Fphse(data_owner_num)
msk = fphse.rsetup()

wk_vec = []
e_tkn_vec = []
st_vec = []
enb_vec = []
b_vec = []


def setup_data_owner(i):
    b_vec.append({})
    wk, st, enb = fphse.wsetup()
    e_tkn, st = fphse.rebuild(i, wk, b_vec[i], st)
    wk_vec.append(wk)
    e_tkn_vec.append(e_tkn)
    st_vec.append(st)
    enb_vec.append(enb)


for i in range(data_owner_num):
    setup_data_owner(i)


def epoch_rotation():
    fphse.edsse.set_epoch(fphse.edsse.get_epoch() + 1)
    for i in range(data_owner_num):
        e_tkn_vec[i], st_vec[i] = fphse.rebuild(i, wk_vec[i], b_vec[i], st_vec[i])

In [4]:
epoch_num = 20
kw_space_size = 100
db_size = 1 * 10 ** 5
db0_size = db_size // data_owner_num
fids = [i for i in range(db0_size)]

r_size = 10 ** 3
# s_size = 100
# r_fids = fids[:r_size]
r_fids = [i for i in range(r_size)]
# fids = fids[r_size:]
fids = fids[:]
r_key = b"r"

i = 0
for j in range(r_size):
    fid = r_fids[j]
    key = r_key
    val = f"{fid}".encode() + b"_" * (val_len - len(f"{fid}"))

    u_no_sse, st_vec[i] = fphse.update_token(i, b_vec[i], wk_vec[i], st_vec[i], OP_ADD, key, val)
    enb_vec[i], e_tkn_vec[i] = fphse.update(u_no_sse, enb_vec[i], e_tkn_vec[i])

i = 1
for key_int in range(kw_space_size):
    key = f"{key_int}".encode()
    val = f"{fid}".encode() + b"_" * (val_len - len(f"{fid}"))

    u_no_sse, st_vec[i] = fphse.update_token(i, b_vec[i], wk_vec[i], st_vec[i], OP_ADD, key, val)
    enb_vec[i], e_tkn_vec[i] = fphse.update(u_no_sse, enb_vec[i], e_tkn_vec[i])
    fid += 1

i = 0
counter = 0
epoch_size = (len(fids) + epoch_num - 1) // epoch_num
print(f"epoch_size: {epoch_size}")
t = time.time()
for epoch in range(epoch_num):
    for _ in range(epoch_size):
        try:
            fid = fids.pop()
        except IndexError:
            break
        key = f"{counter}".encode()
        val = f"{fid}".encode() + b"_" * (val_len - len(f"{fid}"))

        u_no_sse, st_vec[i] = fphse.update_token(i, b_vec[i], wk_vec[i], st_vec[i], OP_ADD, key, val)
        enb_vec[i], e_tkn_vec[i] = fphse.update(u_no_sse, enb_vec[i], e_tkn_vec[i])

        counter += 1
        if counter >= kw_space_size - 1:
            counter = 0

    epoch_rotation()
    print(f"Epoch {epoch} time (s): {time.time() - t}")
    t = time.time()

epoch_size: 10
Epoch 0 time (s): 1.5941760540008545
Epoch 1 time (s): 1.6878151893615723
Epoch 2 time (s): 1.7968876361846924
Epoch 3 time (s): 1.8968517780303955
Epoch 4 time (s): 2.0699946880340576
Epoch 5 time (s): 2.1466615200042725
Epoch 6 time (s): 2.2977445125579834
Epoch 7 time (s): 2.437615394592285
Epoch 8 time (s): 2.607273817062378
Epoch 9 time (s): 2.621516227722168
Epoch 10 time (s): 2.475003242492676
Epoch 11 time (s): 2.531597137451172
Epoch 12 time (s): 2.490351438522339
Epoch 13 time (s): 2.458366870880127
Epoch 14 time (s): 2.502791404724121
Epoch 15 time (s): 2.522726535797119
Epoch 16 time (s): 2.5718185901641846
Epoch 17 time (s): 2.5889413356781006
Epoch 18 time (s): 2.6039626598358154
Epoch 19 time (s): 2.5765693187713623


In [5]:
search_token_t = time.time()
w = r_key
s = [0, 1]
s_no_sse = fphse.search_token(msk, s, w)
print(f"search token (s): {time.time() - search_token_t}")

search_t = time.time()
r, _, _ = fphse.search(s_no_sse, s, enb_vec, e_tkn_vec)
print(f"search (s): {time.time() - search_t}")

assert len(r) == r_size

search token (s): 0.0011334419250488281
Writer 0 time (s): 0.5599400997161865
Writer 1 time (s): 0.5447254180908203
search (s): 1.104928970336914
